# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shakir-j/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Signal check 1 — Volume: I will test whether February GSC impressions are directionally associated with March activity. This is a flag-linked signal because FlyRank's quick-win logic uses traffic volume. Verdict will be based on the observed bucket rates.

Signal check 2 — CTR vs position: I will test whether February click-through rate changes across GSC average-position buckets. This is a flag-linked signal because FlyRank's CTR-fix logic considers CTR relative to position. Verdict will be based on the observed bucket pattern.

Baseline rule: prioritize content that had meaningful February search visibility but weak click engagement. The score combines February impressions and a low-CTR signal, and each scored row receives one reason code and one action label.

In [1]:
# ============================================================
# SECTION 1 — TWO SIGNAL CHECKS
# ============================================================

import duckdb
import pandas as pd
from google.colab import userdata

# ------------------------------------------------------------
# Hugging Face authentication
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise RuntimeError(
        "HF_TOKEN not found. Check Colab Secrets and make sure "
        "the secret is named exactly HF_TOKEN."
    )

# ------------------------------------------------------------
# DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# IMPORTANT: correct DuckDB secret syntax
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")


# ============================================================
# SIGNAL 1 — VOLUME
# ============================================================

volume_check = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions
    FROM {FEB}
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN COALESCE(gsc_clicks, 0)
               + COALESCE(ga4_sessions, 0)
               + COALESCE(scroll_events, 0) > 0
            THEN 1
            ELSE 0
        END AS march_activity_label
    FROM {MAR}
)

SELECT
    CASE
        WHEN feb.gsc_impressions = 0
            THEN '0'
        WHEN feb.gsc_impressions < 100
            THEN '1-99'
        WHEN feb.gsc_impressions < 500
            THEN '100-499'
        WHEN feb.gsc_impressions < 1000
            THEN '500-999'
        ELSE '1000+'
    END AS impressions_bucket,

    COUNT(*) AS n,

    ROUND(
        AVG(COALESCE(mar.march_activity_label, 0)),
        4
    ) AS march_activity_rate

FROM feb

LEFT JOIN mar
    ON feb.client_hash_id = mar.client_hash_id
    AND feb.content_hash_id = mar.content_hash_id

GROUP BY 1

ORDER BY
    CASE impressions_bucket
        WHEN '0' THEN 1
        WHEN '1-99' THEN 2
        WHEN '100-499' THEN 3
        WHEN '500-999' THEN 4
        WHEN '1000+' THEN 5
    END
""").df()

print("\nSIGNAL 1 — VOLUME")
print(volume_check.to_string(index=False))


# ============================================================
# SIGNAL 2 — CTR VS POSITION
# ============================================================

ctr_position_check = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {FEB}
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN COALESCE(gsc_clicks, 0)
               + COALESCE(ga4_sessions, 0)
               + COALESCE(scroll_events, 0) > 0
            THEN 1
            ELSE 0
        END AS march_activity_label
    FROM {MAR}
)

SELECT

    CASE
        WHEN feb.gsc_avg_position IS NULL
            THEN 'no_position'
        WHEN feb.gsc_avg_position <= 10
            THEN '1-10'
        WHEN feb.gsc_avg_position <= 20
            THEN '11-20'
        WHEN feb.gsc_avg_position <= 50
            THEN '21-50'
        ELSE '51+'
    END AS position_bucket,

    CASE
        WHEN feb.gsc_impressions = 0
            THEN 'no_impressions'
        WHEN 100.0 * feb.gsc_clicks / feb.gsc_impressions < 1
            THEN '<1%'
        WHEN 100.0 * feb.gsc_clicks / feb.gsc_impressions < 3
            THEN '1-3%'
        WHEN 100.0 * feb.gsc_clicks / feb.gsc_impressions < 5
            THEN '3-5%'
        ELSE '5%+'
    END AS ctr_bucket,

    COUNT(*) AS n,

    ROUND(
        AVG(COALESCE(mar.march_activity_label, 0)),
        4
    ) AS march_activity_rate

FROM feb

LEFT JOIN mar
    ON feb.client_hash_id = mar.client_hash_id
    AND feb.content_hash_id = mar.content_hash_id

GROUP BY 1, 2

ORDER BY
    CASE position_bucket
        WHEN '1-10' THEN 1
        WHEN '11-20' THEN 2
        WHEN '21-50' THEN 3
        WHEN '51+' THEN 4
        ELSE 5
    END,
    CASE ctr_bucket
        WHEN '<1%' THEN 1
        WHEN '1-3%' THEN 2
        WHEN '3-5%' THEN 3
        WHEN '5%+' THEN 4
        ELSE 5
    END
""").df()

print("\nSIGNAL 2 — CTR VS POSITION")
print(ctr_position_check.to_string(index=False))

print("\nSignal checks completed successfully.")

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026

SIGNAL 1 — VOLUME
impressions_bucket         n  march_activity_rate
                 0 134133828               0.0129
              1-99  64450451               0.0983
           100-499  11389437               0.4588
           500-999   1307427               0.7191
             1000+   3388380               0.1372

SIGNAL 2 — CTR VS POSITION
position_bucket     ctr_bucket         n  march_activity_rate
           1-10            <1%  49312526               0.1534
           1-10           1-3%   2538161               0.4599
           1-10           3-5%    485041               0.3185
           1-10            5%+    588490               0.1599
          11-20            <1%  11737613               0.1651
          11-20           1-3%    414842               0.5615
          11-20           3-5%     94275               0.3412
          11-20            5%+    113712               0.1623
      

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 2 — BUILD THE RANKED BASELINE QUEUE
# ============================================================

# Rule:
# Prioritize February content with meaningful search visibility,
# top-20 average position, and weak CTR below 3%.

# One reason code:
# visible_low_ctr
#
# Action label:
# REVIEW_CTR

import os

OUTPUT_DIR = "work/outputs"
OUTPUT_FILE = f"{OUTPUT_DIR}/baseline_action_score.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Build and rank the baseline queue
# ------------------------------------------------------------

baseline_query = f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE NULL
        END AS ctr_pct

    FROM {FEB}
),

scored AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ctr_pct,

        CASE
            WHEN gsc_impressions >= 100
             AND gsc_avg_position <= 20
             AND ctr_pct < 3
            THEN
                LN(gsc_impressions + 1)
                * (21 - gsc_avg_position)
                * (3 - ctr_pct)

            ELSE 0
        END AS score,

        CASE
            WHEN gsc_impressions >= 100
             AND gsc_avg_position <= 20
             AND ctr_pct < 3
            THEN 'visible_low_ctr'
            ELSE 'not_selected'
        END AS reason_code,

        CASE
            WHEN gsc_impressions >= 100
             AND gsc_avg_position <= 20
             AND ctr_pct < 3
            THEN 'REVIEW_CTR'
            ELSE 'NO_ACTION'
        END AS action_label

    FROM base
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY score DESC,
                 client_hash_id,
                 content_hash_id
    ) AS rank,

    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    ROUND(gsc_avg_position, 2) AS gsc_avg_position,
    ROUND(ctr_pct, 4) AS ctr_pct,
    ROUND(score, 4) AS score,
    reason_code,
    action_label

FROM scored

ORDER BY
    score DESC,
    client_hash_id,
    content_hash_id
"""

# ------------------------------------------------------------
# Write the ranked queue directly to CSV
# ------------------------------------------------------------

con.execute(f"""
COPY (
    {baseline_query}
)
TO '{OUTPUT_FILE}'
(
    FORMAT CSV,
    HEADER TRUE
)
""")

print("Baseline queue written successfully.")
print(f"Output file: {OUTPUT_FILE}")


# ------------------------------------------------------------
# Show the top 10 for inspection
# ------------------------------------------------------------

top10 = con.sql(f"""
    {baseline_query}
    LIMIT 10
""").df()

print("\nTOP 10 BASELINE ACTION QUEUE")
print(top10.to_string(index=False))

print("\nRule:")
print(
    "Prioritize February content with meaningful search visibility, "
    "top-20 average position, and CTR below 3%."
)

print("Reason code: visible_low_ctr")
print("Action label: REVIEW_CTR")


Baseline queue written successfully.
Output file: work/outputs/baseline_action_score.csv

TOP 10 BASELINE ACTION QUEUE
 rank          client_hash_id          content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  ctr_pct    score     reason_code action_label
    1 client_73cda7b4e4f265ea content_fec55986a1868d62            74688           0              0.00   0.0000 706.7767 visible_low_ctr   REVIEW_CTR
    2 client_73cda7b4e4f265ea content_8e1334d6356668e3            62844           0              0.02   0.0000 695.4544 visible_low_ctr   REVIEW_CTR
    3 client_23a62021009f63c4 content_44f34c0a90047651            52631           2              0.02   0.0038 683.4188 visible_low_ctr   REVIEW_CTR
    4 client_73cda7b4e4f265ea content_9c057b66c30a3abb            29079           0              0.00   0.0000 647.4975 visible_low_ctr   REVIEW_CTR
    5 client_73cda7b4e4f265ea content_c9f840183215651b            54024           0              1.26   0.0000 645.3403 visible_low_ctr 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 ranked items using the baseline rule. Each item is marked for CTR review because it has meaningful February search visibility, a top-20 average position, and CTR below 3%. Confidence is moderate because the rule uses only observable February signals; the recommendation could be wrong if the low CTR is explained by query intent, SERP features, tracking issues, or other context not captured by these fields.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 3 — TOP-20 REVIEW
# ============================================================

# Use the ranked queue created in Section 2.
# No March outcome or future-window information is used here.

top20 = con.sql(f"""
    {baseline_query}
    LIMIT 20
""").df()

# ------------------------------------------------------------
# Create one review line for every ranked item
# ------------------------------------------------------------

top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_avg_position",
        "ctr_pct",
        "score",
        "reason_code",
        "action_label"
    ]
].copy()

top20_review["confidence_note"] = (
    "Moderate confidence: meaningful visibility, top-20 position, "
    "and CTR below 3% support the review action."
)

top20_review["what_would_make_it_wrong"] = (
    "The low CTR could be explained by query intent, SERP features, "
    "tracking issues, or context not captured by the February fields."
)

# ------------------------------------------------------------
# Print the review
# ------------------------------------------------------------

print("TOP-20 REVIEW")
print()

for _, row in top20_review.iterrows():

    print(
        f"{int(row['rank'])}. "
        f"Action: {row['action_label']} | "
        f"Reason: {row['reason_code']} | "
        f"Confidence: {row['confidence_note']} | "
        f"Wrong if: {row['what_would_make_it_wrong']}"
    )

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print()
print("Top-20 rows reviewed:", len(top20_review))
print(
    "All actions:",
    top20_review["action_label"].unique().tolist()
)
print(
    "All reason codes:",
    top20_review["reason_code"].unique().tolist()
)

TOP-20 REVIEW

1. Action: REVIEW_CTR | Reason: visible_low_ctr | Confidence: Moderate confidence: meaningful visibility, top-20 position, and CTR below 3% support the review action. | Wrong if: The low CTR could be explained by query intent, SERP features, tracking issues, or context not captured by the February fields.
2. Action: REVIEW_CTR | Reason: visible_low_ctr | Confidence: Moderate confidence: meaningful visibility, top-20 position, and CTR below 3% support the review action. | Wrong if: The low CTR could be explained by query intent, SERP features, tracking issues, or context not captured by the February fields.
3. Action: REVIEW_CTR | Reason: visible_low_ctr | Confidence: Moderate confidence: meaningful visibility, top-20 position, and CTR below 3% support the review action. | Wrong if: The low CTR could be explained by query intent, SERP features, tracking issues, or context not captured by the February fields.
4. Action: REVIEW_CTR | Reason: visible_low_ctr | Confidence: Mo

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some baseline picks may still be weak because the score only uses February search visibility, CTR, and average position. A low CTR can have legitimate explanations such as query intent or SERP features, so the queue should be treated as decision-support rather than a guaranteed recommendation. I also checked the final queue for March, availability, and outcome-derived fields and excluded them from scoring.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# ============================================================

# ------------------------------------------------------------
# 1. Inspect potentially weak baseline picks
# ------------------------------------------------------------

weak_picks = con.sql(f"""
    {baseline_query}
    LIMIT 20
""").df()

# A pick can be considered weaker when it has relatively
# limited search visibility compared with the strongest picks.

weak_picks = weak_picks.sort_values(
    by=["gsc_impressions", "score"],
    ascending=[True, True]
).head(5)

print("WEAK PICK REVIEW")
print("----------------")

for _, row in weak_picks.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"Action={row['action_label']}; "
        f"Reason={row['reason_code']}; "
        f"Impressions={int(row['gsc_impressions'])}; "
        f"Position={row['gsc_avg_position']}; "
        f"CTR={row['ctr_pct']:.4f}%. "
        f"Potential weakness: limited visibility makes the "
        f"CTR-based recommendation less certain."
    )


# ------------------------------------------------------------
# 2. Leakage check
# ------------------------------------------------------------

# These are the actual inputs used by the scoring rule.
score_inputs = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

# Fields that must never be used as scoring inputs.
forbidden_input_terms = [
    "march",
    "label",
    "available",
    "availability",
    "product"
]

forbidden_inputs_found = [
    col for col in score_inputs
    if any(term in col.lower() for term in forbidden_input_terms)
]

print("\nLEAKAGE CHECK")
print("-------------")
print("Score inputs:", score_inputs)
print("Forbidden inputs found:", forbidden_inputs_found)
print("Leakage check passed:", len(forbidden_inputs_found) == 0)

print("\nFuture-window inputs used: 0")
print("Label-derived inputs used: 0")
print("Product/availability inputs used: 0")

# ------------------------------------------------------------
# 3. Verify that the score uses only February inputs
# ------------------------------------------------------------

score_inputs = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("\nScore inputs:", score_inputs)
print("Future-window inputs used: 0")
print("Label-derived inputs used: 0")
print("Product/availability inputs used: 0")


WEAK PICK REVIEW
----------------
Rank 20: Action=REVIEW_CTR; Reason=visible_low_ctr; Impressions=15853; Position=0.02; CTR=0.0000%. Potential weakness: limited visibility makes the CTR-based recommendation less certain.
Rank 19: Action=REVIEW_CTR; Reason=visible_low_ctr; Impressions=16661; Position=0.06; CTR=0.0060%. Potential weakness: limited visibility makes the CTR-based recommendation less certain.
Rank 17: Action=REVIEW_CTR; Reason=visible_low_ctr; Impressions=17226; Position=0.0; CTR=0.0000%. Potential weakness: limited visibility makes the CTR-based recommendation less certain.
Rank 16: Action=REVIEW_CTR; Reason=visible_low_ctr; Impressions=17991; Position=0.03; CTR=0.0000%. Potential weakness: limited visibility makes the CTR-based recommendation less certain.
Rank 18: Action=REVIEW_CTR; Reason=visible_low_ctr; Impressions=18043; Position=0.04; CTR=0.0111%. Potential weakness: limited visibility makes the CTR-based recommendation less certain.

LEAKAGE CHECK
-------------
Sco

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.